In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

In [2]:
import os
from dotenv import load_dotenv
groq_api_key=os.getenv('GROQ_API_KEY')

In [ ]:
loader=PyPDFLoader('')
docs=loader.load()
splits=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
).split_documents(docs)


In [4]:
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vectorstore=Chroma.from_documents(
    documents=splits,
    embedding=embeddings
)


In [5]:
retriever=vectorstore.as_retriever(search_kwargs={"k":4})


In [6]:
from langchain_groq import ChatGroq
llm=ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=groq_api_key
)

In [7]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate


In [8]:
prompt=ChatPromptTemplate.from_template(
    """
    Answer the question based on the context.
    Context:{context}
    Question:{input}
    """
    
)

In [10]:
document_chain=create_stuff_documents_chain(
    llm,
    prompt
)

In [11]:
rag_chain=create_retrieval_chain(
    retriever,
    document_chain
)

In [14]:
response=rag_chain.invoke(
    {
        "input": "Give summary of speech"
    }
)

In [15]:
print(response["answer"])

The speech is the departing address of Dr. A.P.J. Abdul Kalam, the former President of India, after completing his five-year tenure at Rashtrapati Bhavan. He expresses his gratitude to the people of India for their love and support during his term. He shares 10 important messages that he derived from his interactions with people from various walks of life, including:

1. Accelerating development and aspirations of the youth
2. Empowering villages and mobilizing rural core competence
3. Promoting agricultural growth and food security
4. Overcoming problems through partnership and courage
5. Fostering connectivity for societal transformation
6. Defending the nation with pride
7. Encouraging a youth movement for a developed India by 2020

He highlights the importance of value-based education, leadership, and empowerment of the youth. He concludes by thanking the people for their cooperation and support, and reiterates his mission to bring connectivity between the hearts and minds of India